# 06 — CNN-LSTM Baseline Training Training & Evaluation

**Architecture:** Luo et al. Journal of Supercomputing 2023 — CNN+LSTM hybrid  
**Model:** Conv1D(64) → GRU(128, unidirectional) → center → Dense → heads  
**Parameters:** 184K | INT8: 0.184 MB  
**Reference:** Direct predecessor to BiWave-NILM proposed model

**Author:** Chadha Jeddi — NILM Benchmarking Project

In [ ]:
# ---- Setup (same as CNN notebook) ----
import json, sys, os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_NILM = '/content/drive/MyDrive/nilm_project'
REPO_DIR = '/content/nilm-benchmarking'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/chadhajeddi-ux/nilm-benchmarking {REPO_DIR}

os.chdir(REPO_DIR)
for p in ['src','models','models/baselines','models/proposed']:
    sys.path.insert(0, f'{REPO_DIR}/{p}')

for d in ['data/raw/UKDALE','data/raw/REDD','data/raw/AMPds2','data/raw/REFIT','data/processed']:
    os.makedirs(f'{REPO_DIR}/{d}', exist_ok=True)

links = {'data/raw/UKDALE/ukdale.h5': f'{DRIVE_NILM}/data/raw/UKDALE/ukdale.h5',
         'data/raw/REDD/redd.h5': f'{DRIVE_NILM}/data/raw/REDD/redd.h5'}
for local, remote in links.items():
    if os.path.exists(remote) and not os.path.exists(local):
        os.symlink(remote, local)

for folder in ['checkpoints','results']:
    src = f'{DRIVE_NILM}/experiments/{folder}'
    dst = f'{REPO_DIR}/experiments/{folder}'
    os.makedirs(src, exist_ok=True)
    if os.path.exists(dst) and not os.path.islink(dst):
        import shutil; shutil.rmtree(dst)
    if not os.path.islink(dst): os.symlink(src, dst)

proc_src = f'{DRIVE_NILM}/data/processed'
proc_dst = f'{REPO_DIR}/data/processed'
os.makedirs(proc_src, exist_ok=True)
if os.path.exists(proc_dst) and not os.path.islink(proc_dst):
    import shutil; shutil.rmtree(proc_dst)
if not os.path.islink(proc_dst): os.symlink(proc_src, proc_dst)

!pip install -q PyWavelets pyarrow h5py tqdm einops omegaconf torchinfo seaborn

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import warnings; warnings.filterwarnings('ignore')

from config import WINDOW_SIZE, INPUT_CHANNELS, N_APPLIANCES, APPLIANCE_NAMES, APPLIANCES, SEED
from preprocessing import load_ukdale_house, preprocess_house
from dataset import NILMDataset, NormStats, split_train_val, build_dataloaders, load_clean_df, save_clean_df
from metrics import MetricsTracker, multi_task_loss
from train import train_one_epoch, validate_one_epoch, EarlyStopping

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MODEL-SPECIFIC SETTINGS — change only these 3 lines for other notebooks
from cnn_lstm_model import CNNLSTMBaseline as ModelClass
MODEL_NAME = 'cnn_lstm'
LR = 1e-3
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 100
PATIENCE = 15
COLORS = {'kettle':'#D94040','fridge':'#2E9E5A','washing_machine':'#E8922A',
          'dishwasher':'#7B4FBF','microwave':'#CC3399'}
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print(f'Model: {MODEL_NAME} | Device: {DEVICE} | Epochs: {EPOCHS}')

In [ ]:
# ---- Data ----
cached = load_clean_df('UK-DALE', 1)
if cached is not None: clean_df = cached
else:
    raw_df = load_ukdale_house(house=1)
    clean_df = preprocess_house(raw_df)
    save_clean_df(clean_df, 'UK-DALE', 1)

train_df, val_df = split_train_val(clean_df, val_fraction=0.15)
train_loader, val_loader, norm_stats = build_dataloaders(
    train_df, val_df, batch_size=256, train_stride=1,
    val_stride=480, num_workers=2, add_temporal_features=True)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')
x, yp, ys = next(iter(val_loader))
print(f'Batch: x={tuple(x.shape)}, y_power={tuple(yp.shape)}')

In [ ]:
# ---- Model ----
from torchinfo import summary
model = ModelClass(in_channels=INPUT_CHANNELS, window_size=WINDOW_SIZE, n_appliances=N_APPLIANCES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,} | INT8: {n_params/1e6:.3f} MB')
summary(model, input_size=(1, INPUT_CHANNELS, WINDOW_SIZE),
        col_names=['input_size','output_size','num_params'], depth=3)

In [ ]:
# ---- Training ----
import time
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
early_stop = EarlyStopping(patience=PATIENCE)
app_max = {a: float(APPLIANCES[a]['max_power']) for a in APPLIANCE_NAMES}
tracker = MetricsTracker(APPLIANCE_NAMES, app_max)
history = {'epoch':[],'train_loss':[],'val_loss':[],'val_mr':[],'val_f1':[],'val_mae':[],'lr':[]}
best_mr, best_state, best_epoch = -float('inf'), None, 0
start_time = time.time()

for epoch in range(1, EPOCHS+1):
    ep_start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss, metrics = validate_one_epoch(model, val_loader, DEVICE, tracker)
    val_mr = metrics['mean']['mr']; val_f1 = metrics['mean']['f1']; val_mae = metrics['mean']['mae_w']
    cur_lr = optimizer.param_groups[0]['lr']
    for k,v in zip(['epoch','train_loss','val_loss','val_mr','val_f1','val_mae','lr'],
                   [epoch, train_loss, val_loss, val_mr, val_f1, val_mae, cur_lr]):
        history[k].append(v)
    is_best = val_mr > best_mr
    if is_best:
        best_mr=val_mr; best_epoch=epoch
        best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
    print(f"Ep {epoch:3d}/{EPOCHS} | L:{train_loss:.4f}/{val_loss:.4f} | "
          f"MR:{val_mr:.3f} F1:{val_f1:.3f} MAE:{val_mae:.1f}W | "
          f"LR:{cur_lr:.1e} | {time.time()-ep_start:.0f}s{'★' if is_best else ''}")
    scheduler.step(val_mr)
    if early_stop.step(val_mr):
        print(f'Early stopping at epoch {epoch}'); break

total_time = time.time()-start_time
print(f'\nDone in {total_time/60:.1f} min | Best epoch: {best_epoch} | Best MR: {best_mr:.4f}')

In [ ]:
# ---- Training Curves ----
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
epochs = history['epoch']
axes[0,0].plot(epochs, history['train_loss'], label='Train', color='#3366CC')
axes[0,0].plot(epochs, history['val_loss'], label='Val', color='#D94040')
axes[0,0].axvline(best_epoch, color='gray', linestyle=':'); axes[0,0].set_title('Loss'); axes[0,0].legend()
axes[0,1].plot(epochs, history['val_mr'], color='#2E9E5A', linewidth=2)
axes[0,1].axhline(best_mr, color='#2E9E5A', linestyle='--', alpha=0.3, label=f'Best:{best_mr:.3f}')
axes[0,1].set_title('Matching Ratio ↑'); axes[0,1].legend()
axes[1,0].plot(epochs, history['val_f1'], color='#E8922A', linewidth=2); axes[1,0].set_title('F1 Score ↑')
axes[1,1].plot(epochs, history['val_mae'], color='#7B4FBF', linewidth=2); axes[1,1].set_title('MAE (W) ↓')
for ax in axes.flat: ax.axvline(best_epoch, color='gray', linestyle=':', alpha=0.5)
plt.suptitle(f'{MODEL_NAME.upper()} — Training Curves (Best: ep{best_epoch})', fontsize=14)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Final Evaluation ----
model.load_state_dict(best_state); model.to(DEVICE); model.eval()
_, final_metrics = validate_one_epoch(model, val_loader, DEVICE, tracker)
print(f'\n{"="*80}\nFINAL RESULTS — {MODEL_NAME.upper()} (epoch {best_epoch})\n{"="*80}')
tracker.print_table(final_metrics)
results_df = tracker.to_dataframe(final_metrics)
results_df.to_csv(f'experiments/results/{MODEL_NAME}_metrics.csv', index=False)

In [ ]:
# ---- Collect predictions ----
all_pp, all_tp, all_ps, all_ts = [], [], [], []
with torch.no_grad():
    for x, yp, ys in val_loader:
        pp, ps, pg = model(x.to(DEVICE))
        all_pp.append(pp.cpu().numpy()); all_tp.append(yp.numpy())
        all_ps.append((torch.sigmoid(ps)>=0.5).float().cpu().numpy()); all_ts.append(ys.numpy())
pred_power = np.concatenate(all_pp); true_power = np.concatenate(all_tp)
pred_state = np.concatenate(all_ps); true_state = np.concatenate(all_ts)

In [ ]:
# ---- Predictions vs Ground Truth ----
N_SHOW, start_idx = 500, len(pred_power)//3
fig, axes = plt.subplots(N_APPLIANCES, 1, figsize=(16, 3*N_APPLIANCES), sharex=True)
for i, a in enumerate(APPLIANCE_NAMES):
    max_w = APPLIANCES[a]['max_power']
    tw = true_power[start_idx:start_idx+N_SHOW, i]*max_w
    pw = pred_power[start_idx:start_idx+N_SHOW, i]*max_w
    axes[i].plot(tw, color=COLORS[a], alpha=0.7, linewidth=1.2, label='Ground Truth')
    axes[i].plot(pw, color='#333', alpha=0.6, linewidth=0.8, linestyle='--', label='Predicted')
    axes[i].fill_between(range(N_SHOW), tw, pw, alpha=0.15, color='red')
    axes[i].set_ylabel(f'{a}\n(W)', fontsize=10)
    axes[i].text(0.02, 0.85, f'MAE={final_metrics[a]["mae_w"]:.1f}W', transform=axes[i].transAxes,
                 fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    if i==0: axes[i].legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Timestep')
plt.suptitle(f'{MODEL_NAME.upper()} — Predictions vs Ground Truth', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Confusion Matrices ----
fig, axes = plt.subplots(1, N_APPLIANCES, figsize=(4*N_APPLIANCES, 4))
for i, a in enumerate(APPLIANCE_NAMES):
    cm = confusion_matrix(true_state[:,i], pred_state[:,i], labels=[0,1])
    cm_norm = cm.astype('float')/(cm.sum(axis=1, keepdims=True)+1e-8)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=['OFF','ON'], yticklabels=['OFF','ON'],
                ax=axes[i], cbar=False, vmin=0, vmax=1)
    axes[i].set_title(f'{a}\nF1={final_metrics[a]["f1"]:.3f}')
    axes[i].set_xlabel('Predicted')
    if i==0: axes[i].set_ylabel('Actual')
plt.suptitle(f'{MODEL_NAME.upper()} — Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Bar Charts ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = [COLORS[a] for a in APPLIANCE_NAMES]
x_pos = np.arange(N_APPLIANCES)
for ax, metric, label, fmt in zip(axes,
    ['f1','mae_w','mr'], ['F1 Score ↑','MAE (W) ↓','Matching Ratio ↑'],
    ['{:.3f}','{:.1f}','{:.3f}']):
    vals = [final_metrics[a][metric] for a in APPLIANCE_NAMES]
    bars = ax.bar(x_pos, vals, color=colors, alpha=0.85)
    ax.set_xticks(x_pos); ax.set_xticklabels(APPLIANCE_NAMES, rotation=20)
    ax.set_title(label)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val*1.02, fmt.format(val), ha='center', fontsize=9)
plt.suptitle(f'{MODEL_NAME.upper()} — Per-Appliance Results', fontsize=14)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Save all results ----
import json
checkpoint = {'model_name': MODEL_NAME,
    'model_state_dict': best_state, 'best_epoch': best_epoch,
    'best_val_mr': best_mr, 'n_params': n_params,
    'norm_stats': {'agg_mean': norm_stats.agg_mean,
                   'agg_std': norm_stats.agg_std,
                   'appliance_max': norm_stats.appliance_max}}
torch.save(checkpoint, f'experiments/checkpoints/{MODEL_NAME}_best.pth')
history.update({'model': MODEL_NAME, 'training_time_seconds': total_time,
                'best_epoch': best_epoch, 'best_val_mr': best_mr})
with open(f'experiments/results/{MODEL_NAME}_history.json','w') as f: json.dump(history, f, indent=2)
tracker.save_json(f'experiments/results/{MODEL_NAME}_metrics.json', final_metrics, model_name=MODEL_NAME)
norm_stats.save(f'experiments/results/{MODEL_NAME}_norm_stats.json')
print(f'✅ All results saved to Drive!')
print(f'   Best MR: {best_mr:.4f} | Best epoch: {best_epoch} | Time: {total_time/60:.1f} min')